In [ ]:
# ==========================================
# 1 INSTALL LIBRARIES
# ==========================================
!pip install tensorflow pandas scikit-learn -q


# ==========================================
# 2 IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import tensorflow as tf
import glob

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau


# ==========================================
# 3 LOAD ALL YOUR CSV DATASETS
# ==========================================
files = glob.glob("*.csv")

dfs = []

for file in files:
    temp = pd.read_csv(file)
    dfs.append(temp)

df = pd.concat(dfs,ignore_index=True)

print("Total Rows:",len(df))


# ==========================================
# 4 DATA CLEANING
# ==========================================
df = df.drop_duplicates()

df = df.fillna(df.mean(numeric_only=True))

df = df.select_dtypes(include=np.number)

print("Clean Dataset Size:",len(df))


# ==========================================
# 5 FEATURES + TARGET
# ==========================================
X = df.iloc[:,:-1]
y = df.iloc[:,-1]


# ==========================================
# 6 NORMALIZATION
# ==========================================
scaler = StandardScaler()

X = scaler.fit_transform(X)


# ==========================================
# 7 LABEL ENCODING
# ==========================================
encoder = LabelEncoder()

y = encoder.fit_transform(y)


# ==========================================
# 8 TRAIN TEST SPLIT
# ==========================================
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)


# ==========================================
# 9 BUILD OPTIMIZED DEEP MODEL
# ==========================================
model = Sequential([

    Dense(512,activation='relu',input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(256,activation='relu'),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128,activation='relu'),
    Dropout(0.3),

    Dense(len(np.unique(y)),activation='softmax')

])


# ==========================================
# 10 COMPILE MODEL
# ==========================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ==========================================
# 11 OPTIMIZATION
# ==========================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)


# ==========================================
# 12 TRAIN MODEL
# ==========================================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop,reduce_lr]
)


# ==========================================
# 13 EVALUATE MODEL
# ==========================================
loss,accuracy = model.evaluate(X_test,y_test)

print("Final Model Accuracy:",accuracy)


# ==========================================
# 14 SAVE MODEL
# ==========================================
model.save("optimized_crime_model.keras")

print("Model Saved Successfully")

Total Rows: 792
Clean Dataset Size: 504


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.0447 - loss: 3.8566 - val_accuracy: 0.3366 - val_loss: 3.3219 - learning_rate: 5.0000e-04
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2184 - loss: 2.8260 - val_accuracy: 0.7921 - val_loss: 2.9580 - learning_rate: 5.0000e-04
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5558 - loss: 1.8421 - val_accuracy: 0.8614 - val_loss: 2.5969 - learning_rate: 5.0000e-04
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7792 - loss: 1.1603 - val_accuracy: 0.8614 - val_loss: 2.2153 - learning_rate: 5.0000e-04
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8734 - loss: 0.8080 - val_accuracy: 0.8614 - val_loss: 1.8390 - learning_rate: 5.0000e-04
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9181 - loss: 0.5943 - val_accuracy: 0.8614 - val_loss: 1.5056 - learning_rate: 5.0000e-04
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9330 

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import tensorflow as tf
import glob

# =====================
# LOAD MODEL
# =====================
model = tf.keras.models.load_model("optimized_crime_model.keras")

expected_features = model.input_shape[1]

print("Model expects", expected_features, "features")

# =====================
# GET CSV FILES
# =====================
csv_files = glob.glob("*.csv")

print("CSV files detected:", csv_files)

# =====================
# ANALYSIS FUNCTION
# =====================
def analyze_dataset(file_name):
    try:
        df = pd.read_csv(file_name)

        # keep numeric columns
        df = df.select_dtypes(include=np.number)

        if df.shape[1] < expected_features:
            return f"Dataset has only {df.shape[1]} numeric columns but model needs {expected_features}"

        # match feature count
        X = df.iloc[:, :expected_features]

        predictions = model.predict(X)

        classes = np.argmax(predictions, axis=1)

        return f"""
Dataset: {file_name}

Rows: {len(df)}
Columns: {df.shape[1]}

First 10 Predictions:
{classes[:10]}
"""

    except Exception as e:
        return f"Error: {str(e)}"

# =====================
# BUILD GUI
# =====================
interface = gr.Interface(
    fn=analyze_dataset,
    inputs=gr.Dropdown(csv_files, label="Select Crime CSV"),
    outputs="text",
    title="Crime Dataset AI Analyzer",
    description="Select a crime dataset to run prediction"
)

interface.launch(share=True)

Model expects 151 features
CSV files detected: ['trafficing_2016.csv', 'currency_siezed.csv', 'violent_crime_incidence_2016.csv', 'trafficing.csv', 'missing_traced_children.csv', 'murder_motives_2016.csv', 'juvenile_arrested_bckgrnd.csv', 'murder_victims_2016.csv', 'coruption_case.csv', 'violent_crime_incidence.csv', 'missing_traced_children_2016.csv', 'kidnapping.csv', 'murder_motives.csv', 'juvenile_arrested_bckgrnd_2016.csv', 'cyber_crime.csv', 'murder_victims.csv', 'juvenile_crime.csv', 'rape_victims_2016.csv', 'rape_victims.csv', 'kidnapping_2016.csv', 'crime_against_children.csv', 'crimes.csv']
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://979383d36de6520313.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("optimized_crime_model.keras")

feature_count = model.input_shape[1]

def predict_crime(*inputs):

    data = np.array(inputs).reshape(1, -1)

    prediction = model.predict(data)
    class_index = np.argmax(prediction)

    return f"🚨 Predicted Class: {class_index}"

inputs = [gr.Number(label=f"Feature {i+1}") for i in range(feature_count)]

interface = gr.Interface(
    fn=predict_crime,
    inputs=inputs,
    outputs="text",
    title="Crime Prediction AI",
    description="Enter feature values"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b18c76b197e05289ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
